# Сравнение входов BM25

Четыре варианта на одном сплите и одинаковых тестовых контекстах. Корпус — уникальные объявления из всего `train.parquet`, включая объявления отложенных взаимодействий. Для ID с несколькими версиями полей берём последнюю строку файла.

Recall@K реализован ниже вручную: для каждого контекста делим число найденных уникальных наблюдаемых объявлений на число его уникальных наблюдаемых объявлений, затем усредняем по всем контекстам. Запрос без выдачи получает 0. Это восстановление наблюдаемых пар, а не оценка по полной разметке релевантности.

In [1]:
from pathlib import Path
import sys
import gc

import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/split.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.split import split_queries
from src.retrievers.bm25 import BM25Retriever

SEED = 42
K = 10
N_TEST_QUERIES = 1000  # None — все контексты test
QUERY_COLUMNS = [
    "search_query", "search_location_id", "search_is_delivery_search",
    "search_infm_params_text", "search_category",
]
ITEM_COLUMNS = ["item_id", "item_title_raw", "item_description_raw", "item_infm_params_text"]
results = {}

## Данные и сплит

In [2]:
data = pl.read_parquet(ROOT / "data/train.parquet", columns=QUERY_COLUMNS + ["item_id"])
data = data.with_columns(pl.Series("split", split_queries(data["search_query"], seed=SEED)))
items = (
    pl.scan_parquet(ROOT / "data/train.parquet").select(ITEM_COLUMNS)
    .unique(subset="item_id", keep="last", maintain_order=True)
    .sort("item_id").collect()
)
test = (
    data.filter(pl.col("split") == "test")
    .group_by(QUERY_COLUMNS).agg(pl.col("item_id").unique().alias("targets"))
    .sort(QUERY_COLUMNS)
)
if N_TEST_QUERIES is not None:
    test = test.sample(n=min(N_TEST_QUERIES, len(test)), seed=SEED, shuffle=True)
queries = pd.DataFrame(test.select(QUERY_COLUMNS).to_dict(as_series=False))
targets = [set(ids) for ids in test["targets"].to_list()]
assert all(targets)
assert set().union(*targets).issubset(set(items["item_id"]))
print(f"Корпус: {len(items):,} объявлений; тестовая выборка: {len(queries):,} контекстов")
display(data.group_by("split").len().sort("split"))

Корпус: 344,825 объявлений; тестовая выборка: 1,000 контекстов


split,len
str,u32
"""test""",83218
"""train""",360319
"""val""",54136


## Recall@K

In [3]:
def recall_at_k(candidates, targets, k):
    retrieved = (
        candidates.loc[candidates["rank"] <= k]
        .groupby("query_index")["item_id"].agg(set).to_dict()
    )
    return float(np.mean([
        len(relevant & retrieved.get(i, set())) / len(relevant)
        for i, relevant in enumerate(targets)
    ]))

# Дубли не увеличивают Recall; второй запрос без выдачи даёт 0.
example = pd.DataFrame({"query_index": [0, 0, 0], "item_id": ["a", "a", "b"], "rank": [1, 2, 3]})
assert recall_at_k(example, [{"a", "b"}, {"c"}], k=2) == 0.25
assert recall_at_k(example.iloc[:0], [{"a"}], k=2) == 0.0

## 1. Запрос → заголовок

### Создание модели

In [4]:
model = BM25Retriever(query_text_column="search_query", item_text_column="item_title_raw", lemmatize=False, batch_size=16)

### Создание корпуса текстов

In [5]:
corpus = pd.DataFrame(items.select("item_id", "item_title_raw").to_dict(as_series=False))

### Прогон на тесте

In [6]:
model.fit(corpus)
candidates = model.retrieve(queries, k=K)
display(candidates.head())

,query_index,item_id,rank,score
0,0,9867422816e3c145,1,5.695249
1,0,435b6c46fbc72c27,2,5.695249
2,0,b356572ac4f216be,3,5.695249
3,0,0302980b2f1874dd,4,5.695249
4,0,29b354b0222fc8cd,5,5.695249


### Подсчёт метрик

In [7]:
results["Запрос → заголовок"] = recall_at_k(candidates, targets, K)
print(f"Recall@{K}: {results['Запрос → заголовок']:.6f}")
del model, corpus, candidates
gc.collect();

Recall@10: 0.061746


## 2. Параметры запроса → параметры объявления

Пустые параметры запроса сохраняются в оценке: отсутствие кандидатов даёт Recall = 0.

### Создание модели

In [8]:
model = BM25Retriever(query_text_column="search_infm_params_text", item_text_column="item_infm_params_text", lemmatize=False, batch_size=16)

### Создание корпуса текстов

In [9]:
corpus = pd.DataFrame(items.select("item_id", "item_infm_params_text").to_dict(as_series=False))

### Прогон на тесте

In [10]:
model.fit(corpus)
candidates = model.retrieve(queries, k=K)
display(candidates.head())

,query_index,item_id,rank,score
0,1,98d8be92224f11ad,1,5.633627
1,1,814673e3258e1578,2,5.633627
2,1,995f9e1eae75fa2d,3,5.633627
3,1,953c249ad20b9d5e,4,5.633627
4,1,f4d352e43647b4e6,5,5.633627


### Подсчёт метрик

In [11]:
results["Параметры запроса → параметры объявления"] = recall_at_k(candidates, targets, K)
print(f"Recall@{K}: {results['Параметры запроса → параметры объявления']:.6f}")
del model, corpus, candidates
gc.collect();

Recall@10: 0.002833


## 3. Запрос → описание

### Создание модели

In [12]:
model = BM25Retriever(query_text_column="search_query", item_text_column="item_description_raw", lemmatize=False, batch_size=16)

### Создание корпуса текстов

In [13]:
corpus = pd.DataFrame(items.select("item_id", "item_description_raw").to_dict(as_series=False))

### Прогон на тесте

In [14]:
model.fit(corpus)
candidates = model.retrieve(queries, k=K)
display(candidates.head())

,query_index,item_id,rank,score
0,0,8e533c8eea3de5ff,1,8.272605
1,0,c5c957ae148ea56f,2,8.147454
2,0,5d1b0926e499d663,3,8.053246
3,0,84918a0cbab4ea6e,4,8.015007
4,0,17093538e3a9acd4,5,8.014242


### Подсчёт метрик

In [15]:
results["Запрос → описание"] = recall_at_k(candidates, targets, K)
print(f"Recall@{K}: {results['Запрос → описание']:.6f}")
del model, corpus, candidates
gc.collect();

Recall@10: 0.039200


## 4. Запрос → заголовок + описание

### Создание модели

In [16]:
model = BM25Retriever(query_text_column="search_query", item_text_column="item_text", lemmatize=False, batch_size=16)

### Создание корпуса текстов

In [17]:
corpus = pd.DataFrame(items.select(
    "item_id",
    pl.concat_str([
        pl.col("item_title_raw").fill_null(""),
        pl.col("item_description_raw").fill_null(""),
    ], separator=" ").alias("item_text"),
).to_dict(as_series=False))

### Прогон на тесте

In [18]:
model.fit(corpus)
candidates = model.retrieve(queries, k=K)
display(candidates.head())

,query_index,item_id,rank,score
0,0,aa4e56c6612043df,1,8.423801
1,0,8e533c8eea3de5ff,2,8.120924
2,0,c5c957ae148ea56f,3,7.939265
3,0,5d1b0926e499d663,4,7.849737
4,0,84918a0cbab4ea6e,5,7.814640


### Подсчёт метрик

In [19]:
results["Запрос → заголовок + описание"] = recall_at_k(candidates, targets, K)
print(f"Recall@{K}: {results['Запрос → заголовок + описание']:.6f}")
del model, corpus, candidates
gc.collect();

Recall@10: 0.060700


## Сравнение

In [20]:
pd.DataFrame.from_dict(results, orient="index", columns=[f"Recall@{K}"]).rename_axis("Вход BM25").sort_values(f"Recall@{K}", ascending=False)

,Recall@10
Вход BM25,
Запрос → заголовок,0.061746
Запрос → заголовок + описание,0.060700
Запрос → описание,0.039200
Параметры запроса → параметры объявления,0.002833
